In [ ]:
!pip install --upgrade binance-historical-data plotly -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 272.8/272.8 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 19.4 MB/s eta 0:00:00


In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta, date, time
from binance_historical_data import BinanceDataDumper
warnings.filterwarnings('ignore')

In [ ]:
def create_data_dumper(market_type="spot", data_frequency="1h"):
    return BinanceDataDumper(
        path_dir_where_to_dump=".",
        asset_class=market_type,
        data_type="klines",
        data_frequency=data_frequency,
    )

def get_list_all_trading_pairs(market_type="spot"):
    data_dumper = create_data_dumper(market_type)
    return data_dumper.get_list_all_trading_pairs()

def find_first_data_date(ticker, market_type="spot"):
    data_dumper = create_data_dumper(market_type)
    return data_dumper.get_min_start_date_for_ticker(ticker)

def filter_usdt_tickers(tickers):
    exclude_keywords = ["UPUSDT", "DOWNUSDT", "BEARUSDT", "BULLUSDT"]
    return [ticker for ticker in tickers if ticker.endswith("USDT") and not any(ex in ticker for ex in exclude_keywords)]

def find_first_data_date(ticker):
    data_dumper = BinanceDataDumper(
        path_dir_where_to_dump=".",
        asset_class="spot",
        data_type="klines",
        data_frequency="1h",
    )
    return data_dumper.get_min_start_date_for_ticker(ticker)

def format_time(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"

def detect_timestamp_unit(timestamp):
    num_digits = len(str(timestamp))
    if num_digits == 13:
        return 'ms'
    elif num_digits == 16:
        return 'us'
    else:
        raise ValueError(f"Timestamp không hợp lệ: {timestamp}")

def convert_timestamp(timestamp):
    unit = detect_timestamp_unit(timestamp)
    return pd.to_datetime(timestamp, unit=unit, errors='coerce')

def download_ticker(ticker, date_start, date_end, data_frequency="1h", market_type="spot"):
    data_dumper = create_data_dumper(market_type, data_frequency)

    date_start = datetime.strptime(date_start, "%Y-%m-%d").date()
    date_end = datetime.strptime(date_end, "%Y-%m-%d").date()

    data_dumper.dump_data(
        tickers=ticker,
        date_start=date_start,
        date_end=date_end,
        is_to_update_existing=False,
    )

def read_csv_file(file_path):
    df = pd.read_csv(file_path)
    df.columns = [
        "open_time", "Open", "High", "Low", "Close", "volume",
        "close_time", "quote_asset_volume", "number_of_trades",
        "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume", "ignore"
    ]
    df['open_time'] = df['open_time'].apply(convert_timestamp)
    df['close_time'] = df['close_time'].apply(convert_timestamp)
    return df

def get_csv_files(directory):
    try:
        if not os.path.exists(directory):
            print(f"Warning: Thư mục không tồn tại: {directory}")
            return []
        return [os.path.join(directory, f) for f in os.listdir(directory) if f.endswith('.csv')]
    except Exception as e:
        print(f"Lỗi khi đọc thư mục {directory}: {str(e)}")
        return []

# Load data from file
def process_csv_files(ticker, data_frequency="1h", market_type="spot"):
    base_path = f"{market_type}"

    daily_path = os.path.join(os.getcwd(), f"{base_path}/daily/klines/{ticker}/{data_frequency}")
    monthly_path = os.path.join(os.getcwd(), f"{base_path}/monthly/klines/{ticker}/{data_frequency}")

    daily_files = get_csv_files(daily_path)
    monthly_files = get_csv_files(monthly_path)

    all_files = daily_files + monthly_files

    if not all_files:
        print(f"❗ Không có file CSV nào cho {ticker} ({market_type})")
        return None

    data = pd.concat([read_csv_file(file) for file in all_files], ignore_index=True)
    data.sort_values(by='open_time', inplace=True)

    return data

In [ ]:
ticker = 'BTCUSDT'
data_frequency = '1m'
download_ticker(ticker, "2026-04-01", "2026-04-16", data_frequency, market_type="spot")
data = process_csv_files(ticker, data_frequency)
data = data.drop(columns=['close_time',	'quote_asset_volume',	'number_of_trades',	'taker_buy_base_asset_volume',	'taker_buy_quote_asset_volume',	'ignore'])
data


---> Found overall tickers: 620
---> Filter to asked tickers: 7
------> Tickers left: 2
Download full data for 2 tickers: 
---> Data will be saved here: /content/spot
---> Data Frequency: 1m
---> Start Date: 20260401
---> End Date: 20260416


Tickers:   0%|          | 0/2 [00:00<?, ?it/s]

daily files to download:   0%|          | 0/16 [00:00<?, ?files/s]

daily files to download:   0%|          | 0/16 [00:00<?, ?files/s]

Tried to dump data for 2 tickers:
---> For BTCUSDT new data saved for: 0 months 16 days
---> For BTCUSD new data saved for: 0 months 16 days


,open_time,Open,High,Low,Close,volume
15829,2026-04-01 00:01:00,68218.36,68218.36,68120.99,68164.79,18.83543
15830,2026-04-01 00:02:00,68164.79,68177.42,68142.66,68177.41,6.56973
15831,2026-04-01 00:03:00,68177.40,68227.01,68177.40,68196.00,8.69624
15832,2026-04-01 00:04:00,68196.01,68203.20,68190.52,68197.33,4.17840
15833,2026-04-01 00:05:00,68197.34,68199.10,68143.74,68163.41,3.55103
...,...,...,...,...,...,...
23019,2026-04-16 23:55:00,75051.02,75062.16,75051.01,75062.16,0.64683
23020,2026-04-16 23:56:00,75062.15,75120.85,75051.02,75111.24,8.77687
23021,2026-04-16 23:57:00,75111.25,75141.35,75090.61,75141.35,7.52782
23022,2026-04-16 23:58:00,75141.35,75159.25,75141.34,75159.24,6.49276


In [ ]:
df = data.copy()

# OHLC Chart
fig = go.Figure(
    data=[
        go.Candlestick(
            x=df["open_time"],
            open=df["Open"],
            high=df["High"],
            low=df["Low"],
            close=df["Close"],
            name="OHLC"
        )
    ]
)

# Layout config
fig.update_layout(
    title="OHLC Chart",
    xaxis_title="Time",
    yaxis_title="Price",
    xaxis_rangeslider_visible=False,
    template="plotly_dark",
    height=600
)

fig.show()

# Submission

In [ ]:
import shutil
import random
import plotly.express as px
from tqdm.auto import tqdm

In [ ]:
cache_dir = "/content/spot"

if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print("Cache deleted:", cache_dir)
else:
    print("No cache found")

Cache deleted: /content/spot


In [ ]:
tickers = [
    "BTCUSDT","ETHUSDT","XRPUSDT","BNBUSDT",
    "SOLUSDT","TRXUSDT","DOGEUSDT",
    "BCHUSDT","ADAUSDT","XLMUSDT"
]

START = "2021-04-01"
END   = "2026-05-02"

if os.path.exists("spot"):
    shutil.rmtree("spot")

for t in tickers:
    print("Downloading:", t)
    download_ticker(t, START, END, data_frequency="1d")

Downloading: BTCUSDT
---> Found overall tickers: 620
---> Filter to asked tickers: 7
------> Tickers left: 2
Download full data for 2 tickers: 
---> Data will be saved here: /content/spot
---> Data Frequency: 1d
---> Start Date: 20210401
---> End Date: 20260430


Tickers:   0%|          | 0/2 [00:00<?, ?it/s]

monthly files to download:   0%|          | 0/60 [00:00<?, ?files/s]

daily files to download:   0%|          | 0/30 [00:00<?, ?files/s]

monthly files to download:   0%|          | 0/4 [00:00<?, ?files/s]

...

monthly files to download:   0%|          | 0/60 [00:00<?, ?files/s]

daily files to download:   0%|          | 0/30 [00:00<?, ?files/s]

monthly files to download: 0files [00:00, ?files/s]

daily files to download: 0files [00:00, ?files/s]

Tried to dump data for 2 tickers:
---> For XLMUSDT new data saved for: 60 months 30 days
---> For XLMUSD new data saved for: 0 months 0 days


In [ ]:
def load_full_data(tickers, freq="1d"):
    data = {}
    for t in tickers:
        df = process_csv_files(t, freq)
        if df is None or len(df) == 0:
            continue

        df = df[["open_time","Open","High","Low","Close","volume"]]
        df = df.set_index("open_time").sort_index()

        data[t] = df
    return data


def align_data(data):
    idx = sorted(set().union(*[df.index for df in data.values()]))

    out = {}
    for t, df in data.items():
        out[t] = df.reindex(idx).ffill()

    return out


def compute_returns(data):
    prices = pd.concat([df["Close"].rename(t) for t, df in data.items()], axis=1)
    prices = prices.ffill()

    ret = np.log(prices).diff()
    return ret.dropna()

In [ ]:
PERIODS = [3, 7, 14, 21, 30, 60, 91, 182, 273, 365]
TOP_K_LIST = [1, 2, 3, 4, 5]
NORMALIZATION_METHOD = "zscore"   # "rank" or "zscore"

def compute_features(df):

    df = df.copy()

    # ===== RETURNS =====
    df["ret"] = np.log(df["Close"]).diff()

    # ===== CANDLE STRUCTURE =====
    df["hl_range"] = (df["High"] - df["Low"]) / df["Close"]
    df["body"] = (df["Close"] - df["Open"]) / df["Open"]

    df["upper_wick"] = (df["High"] - df[["Open","Close"]].max(axis=1)) / df["Close"]
    df["lower_wick"] = (df[["Open","Close"]].min(axis=1) - df["Low"]) / df["Close"]

    df["close_pos"] = (df["Close"] - df["Low"]) / (df["High"] - df["Low"] + 1e-12)

    vol = df["volume"]

    for p in PERIODS:

        # ===== CORE =====
        df[f"ret_sum_{p}"] = df["ret"].rolling(p).sum()
        df[f"rv_{p}"] = df["ret"].rolling(p).std()

        df[f"mom_z_{p}"] = (
            df[f"ret_sum_{p}"] - df[f"ret_sum_{p}"].rolling(p).mean()
        ) / (df[f"ret_sum_{p}"].rolling(p).std() + 1e-12)

        # ===== VOLUME =====
        df[f"vol_z_{p}"] = (vol - vol.rolling(p).mean()) / (vol.rolling(p).std() + 1e-12)

        # ===== RANGE =====
        high = df["High"].rolling(p).max()
        low  = df["Low"].rolling(p).min()

        df[f"range_pos_{p}"] = (df["Close"] - low) / (high - low + 1e-12)

        high_c = df["Close"].rolling(p).max()
        low_c  = df["Close"].rolling(p).min()

        df[f"close_range_{p}"] = (df["Close"] - low_c) / (high_c - low_c + 1e-12)

        # ===== ROLLING CANDLE =====
        df[f"avg_hl_{p}"] = df["hl_range"].rolling(p).mean()
        df[f"avg_body_{p}"] = df["body"].rolling(p).mean()
        df[f"avg_up_{p}"] = df["upper_wick"].rolling(p).mean()
        df[f"avg_low_{p}"] = df["lower_wick"].rolling(p).mean()
        df[f"avg_close_pos_{p}"] = df["close_pos"].rolling(p).mean()

    # ===== ADVANCED FEATURES =====

    # ===== TREND (EMA STRUCTURE) =====
    for p in PERIODS:
        ema_fast = df["Close"].ewm(span=max(2, p//2)).mean()
        ema_slow = df["Close"].ewm(span=p).mean()

        df[f"ema_spread_{p}"] = (ema_fast - ema_slow) / (df["Close"] + 1e-12)

    # ===== VOL OF VOL =====
    for p in PERIODS:
        df[f"vol_of_vol_{p}"] = df[f"rv_{p}"].rolling(p).std()

    # ===== RANGE COMPRESSION / EXPANSION =====
    for p in PERIODS:
        df[f"range_compression_{p}"] = (
            df["hl_range"] / (df["hl_range"].rolling(p).mean() + 1e-12)
        )

    # ===== RETURN SKEWNESS =====
    for p in PERIODS:
        df[f"ret_skew_{p}"] = df["ret"].rolling(p).skew()

    # ===== RETURN KURTOSIS =====
    for p in PERIODS:
        df[f"ret_kurt_{p}"] = df["ret"].rolling(p).kurt()

    # ===== ENTROPY PROXY =====
    for p in PERIODS:
        prob_up = (df["ret"] > 0).rolling(p).mean()
        df[f"entropy_{p}"] = -(prob_up * np.log(prob_up + 1e-12) +
                              (1 - prob_up) * np.log(1 - prob_up + 1e-12))

    # ===== LIQUIDITY SHOCK =====
    for p in PERIODS:
        df[f"liq_shock_{p}"] = df["volume"] / (df["volume"].rolling(p).mean() + 1e-12)

    # ===== PRICE EFFICIENCY (trend vs noise) =====
    for p in PERIODS:
        net = abs(df["Close"] - df["Close"].shift(p))
        gross = df["Close"].diff().abs().rolling(p).sum()
        df[f"efficiency_{p}"] = net / (gross + 1e-12)

    # ===== NONLINEAR TRANSFORMS =====
    for p in PERIODS:
        df[f"mom_squared_{p}"] = df[f"ret_sum_{p}"] ** 2
        df[f"mom_cubed_{p}"] = df[f"ret_sum_{p}"] ** 3

    return df

SIGNAL_LIBRARY = {

    # ===== CORE =====
    "momentum": lambda df, p: df[f"mom_z_{p}"],
    "meanrev": lambda df, p: -df[f"ret_sum_{p}"],

    # ===== VOLUME =====
    "volume_thrust": lambda df, p: df[f"vol_z_{p}"] * df[f"ret_sum_{p}"],

    # ===== PRICE STRUCTURE =====
    "range_reversion": lambda df, p: df[f"close_range_{p}"],
    "range_momentum": lambda df, p: df[f"range_pos_{p}"],

    # ===== VOL REGIME =====
    "vol_regime": lambda df, p: (
        df[f"rv_{p}"] - df[f"rv_{p}"].rolling(2*p).mean()
    ) / (df[f"rv_{p}"].rolling(2*p).std() + 1e-12),

    # ===== CANDLE =====
    "wick_ratio": lambda df, p: (
        df[f"avg_up_{p}"] - df[f"avg_low_{p}"]
    ) / (df[f"avg_up_{p}"] + df[f"avg_low_{p}"] + 1e-12),

    # ===== VOLUME + PRICE =====
    "volume_pressure": lambda df, p: (
        df[f"vol_z_{p}"] * (df["Close"] - df["Open"]) /
        (df["High"] - df["Low"] + 1e-12)
    ),

    "acc_dist": lambda df, p: (
        df["volume"] *
        ((df["Close"] - df["Low"]) - (df["High"] - df["Close"])) /
        (df["High"] - df["Low"] + 1e-12)
    ).rolling(p).mean(),

    # ===== MOMENTUM VARIANTS =====
    "rel_momentum": lambda df, p: (
        df[f"ret_sum_{p}"] /
        (df[f"ret_sum_{p}"].abs().rolling(p).mean() + 1e-12)
    ),

    "price_accel": lambda df, p: (
        (df["Close"].diff(p) / (df["Close"].shift(p) + 1e-12))
        .diff().rolling(p).mean()
    ),

    # ===== GAP =====
    "close_pos_change": lambda df, p: df[f"avg_close_pos_{p}"].diff(p),

    # ===== TREND STRUCTURE =====
    "ema_trend": lambda df, p: df[f"ema_spread_{p}"],

    # ===== VOL DYNAMICS =====
    "vol_of_vol": lambda df, p: df[f"vol_of_vol_{p}"],

    # ===== RANGE REGIME =====
    "range_compression": lambda df, p: df[f"range_compression_{p}"],

    # ===== HIGHER MOMENTS =====
    "skewness": lambda df, p: df[f"ret_skew_{p}"],
    "kurtosis": lambda df, p: df[f"ret_kurt_{p}"],

    # ===== ENTROPY =====
    "entropy": lambda df, p: df[f"entropy_{p}"],

    # ===== LIQUIDITY =====
    "liq_shock": lambda df, p: df[f"liq_shock_{p}"],

    # ===== EFFICIENCY =====
    "efficiency": lambda df, p: df[f"efficiency_{p}"],

    # ===== NONLINEAR MOMENTUM =====
    "mom_quadratic": lambda df, p: df[f"mom_squared_{p}"],
    "mom_cubic": lambda df, p: df[f"mom_cubed_{p}"],

    # ===== INTERACTIONS (🔥 VERY IMPORTANT) =====
    "mom_x_range": lambda df, p: df[f"ret_sum_{p}"] * df[f"range_pos_{p}"],
    "vol_x_range": lambda df, p: df[f"vol_z_{p}"] * df[f"range_pos_{p}"],

    # ===== REGIME SWITCHING =====
    "trend_vol_regime": lambda df, p: df[f"ret_sum_{p}"] / (df[f"rv_{p}"] + 1e-12),
    "range_vol_regime": lambda df, p: df[f"range_pos_{p}"] / (df[f"rv_{p}"] + 1e-12),



    # ===== 1. VOLATILITY ASYMMETRY =====
    "vol_up_down_asym": lambda df, p: (
        df["ret"].clip(lower=0).rolling(p).std()
        - df["ret"].clip(upper=0).abs().rolling(p).std()
    ),

    # ===== 2. TREND STABILITY =====
    "trend_stability": lambda df, p: (
        df["ret_sum_{p}".format(p=p)] /
        (df["ret"].abs().rolling(p).sum() + 1e-12)
    ),

    # ===== 4. SHOCK RECOVERY =====
    "shock_recovery": lambda df, p: (
        df["ret"].rolling(p).min().abs() /
        (df["ret"].rolling(p).mean().abs() + 1e-12)
    ),

    # ===== 5. RANGE BREAK INTENSITY =====
    "break_intensity": lambda df, p: (
        (df["Close"] > df["High"].rolling(p).max().shift(1)).astype(int)
        - (df["Close"] < df["Low"].rolling(p).min().shift(1)).astype(int)
    ),

    # ===== 6. VOL CLUSTERING =====
    "vol_cluster": lambda df, p: (
        df["rv_{p}".format(p=p)].rolling(p).corr(df["rv_{p}".format(p=p)].shift(1))
    ),

    # ===== 7. MICRO TREND FLIP RATE =====
    "flip_rate": lambda df, p: (
        np.sign(df["ret"]).diff().abs().rolling(p).mean()
    ),

    # ===== 10. FRACTAL ROUGHNESS PROXY =====
    "roughness": lambda df, p: (
        df["Close"].diff().abs().rolling(p).sum() /
        (abs(df["Close"] - df["Close"].shift(p)) + 1e-12)
    ),
}

STATIC_SIGNALS = {
    # "mom_accel": lambda df: df["ret"].diff()
}

def build_signals(data):

    out = {}

    for t, df in data.items():

        df = compute_features(df)

        sigs = {}

        # ===== PERIOD-BASED =====
        for name, fn in SIGNAL_LIBRARY.items():
            for p in PERIODS:

                try:
                    s = fn(df, p)
                    sigs[f"{name}_{p}"] = s
                except Exception:
                    continue  # robust to edge cases

        # ===== STATIC =====
        for name, fn in STATIC_SIGNALS.items():
            sigs[name] = fn(df)

        out[t] = pd.concat(sigs, axis=1)

    return out

In [ ]:
# LOCKED_SIGNALS = {

#     # ===== TOP PERFORMERS =====
#     "momentum_7": lambda df: df["mom_z_7"],
#     "rel_momentum_3": lambda df: df["ret_sum_3"] / (df["ret_sum_3"].abs().rolling(3).mean() + 1e-12),
#     "range_reversion_21": lambda df: df["close_range_21"],
#     "volume_thrust_91": lambda df: df["vol_z_91"] * df["ret_sum_91"],
#     "range_momentum_3": lambda df: df["range_pos_3"],
#     "mom_x_range_3": lambda df: df["ret_sum_3"] * df["range_pos_3"],
#     "range_vol_regime_21": lambda df: df["range_pos_21"] / (df["rv_21"] + 1e-12),

#     # ===== MEAN REVERSION / CANDLE =====
#     "wick_ratio_60": lambda df: (
#         df["avg_up_60"] - df["avg_low_60"]
#     ) / (df["avg_up_60"] + df["avg_low_60"] + 1e-12),

#     "meanrev_3": lambda df: -df["ret_sum_3"],

#     # ===== NONLINEAR MOMENTUM =====
#     "mom_cubic_3": lambda df: df["mom_cubed_3"],
#     "mom_quadratic_7": lambda df: df["mom_squared_7"],

#     # ===== DISTRIBUTION =====
#     "skewness_91": lambda df: df["ret_skew_91"],
#     "kurtosis_14": lambda df: df["ret_kurt_14"],

#     # ===== TREND =====
#     "ema_trend_3": lambda df: df["ema_spread_3"],
#     "trend_vol_regime_7": lambda df: df["ret_sum_7"] / (df["rv_7"] + 1e-12),

#     # ===== INTERACTIONS =====
#     "vol_x_range_60": lambda df: df["vol_z_60"] * df["range_pos_60"],

#     # ===== REGIME / ENTROPY =====
#     "entropy_365": lambda df: df["entropy_365"],
#     "vol_regime_273": lambda df: (
#         df["rv_273"] - df["rv_273"].rolling(546).mean()
#     ) / (df["rv_273"].rolling(546).std() + 1e-12),

#     # ===== STRUCTURE =====
#     "efficiency_91": lambda df: df["efficiency_91"],
#     "range_compression_30": lambda df: df["range_compression_30"],

#     # ===== FLOW =====
#     "acc_dist_7": lambda df: (
#         df["volume"] *
#         ((df["Close"] - df["Low"]) - (df["High"] - df["Close"])) /
#         (df["High"] - df["Low"] + 1e-12)
#     ).rolling(7).mean(),

#     "volume_pressure_3": lambda df: (
#         df["vol_z_3"] * (df["Close"] - df["Open"]) /
#         (df["High"] - df["Low"] + 1e-12)
#     ),

#     # ===== DYNAMICS =====
#     "close_pos_change_91": lambda df: df["avg_close_pos_91"].diff(91),
#     "price_accel_30": lambda df: (
#         (df["Close"].diff(30) / (df["Close"].shift(30) + 1e-12))
#         .diff().rolling(30).mean()
#     ),

#     # ===== VOL DYNAMICS =====
#     "vol_of_vol_182": lambda df: df["vol_of_vol_182"],

#     # ===== LIQUIDITY =====
#     "liq_shock_7": lambda df: df["liq_shock_7"],
# }


# NEW_SIGNAL_LIBRARY = {

#     # ===== 1. VOLATILITY ASYMMETRY =====
#     "vol_up_down_asym": lambda df, p: (
#         df["ret"].clip(lower=0).rolling(p).std()
#         - df["ret"].clip(upper=0).abs().rolling(p).std()
#     ),

#     # ===== 2. TREND STABILITY =====
#     "trend_stability": lambda df, p: (
#         df["ret_sum_{p}".format(p=p)] /
#         (df["ret"].abs().rolling(p).sum() + 1e-12)
#     ),

#     # ===== 3. RETURN PATH CONVEXITY =====
#     "path_convexity": lambda df, p: (
#         df["ret"].rolling(p).apply(
#             lambda x: np.sum(np.diff(x, 2)), raw=True
#         )
#     ),

#     # ===== 4. SHOCK RECOVERY =====
#     "shock_recovery": lambda df, p: (
#         df["ret"].rolling(p).min().abs() /
#         (df["ret"].rolling(p).mean().abs() + 1e-12)
#     ),

#     # ===== 5. RANGE BREAK INTENSITY =====
#     "break_intensity": lambda df, p: (
#         (df["Close"] > df["High"].rolling(p).max().shift(1)).astype(int)
#         - (df["Close"] < df["Low"].rolling(p).min().shift(1)).astype(int)
#     ),

#     # ===== 6. VOL CLUSTERING =====
#     "vol_cluster": lambda df, p: (
#         df["rv_{p}".format(p=p)].rolling(p).corr(df["rv_{p}".format(p=p)].shift(1))
#     ),

#     # ===== 7. MICRO TREND FLIP RATE =====
#     "flip_rate": lambda df, p: (
#         np.sign(df["ret"]).diff().abs().rolling(p).mean()
#     ),

#     # ===== 8. RANGE SKEW =====
#     "range_skew": lambda df, p: (
#         (df["High"] - df["Close"]) - (df["Close"] - df["Low"])
#     ).rolling(p).mean(),

#     # ===== 9. PRICE ACCELERATION ASYMMETRY =====
#     "accel_asym": lambda df, p: (
#         df["Close"].diff().clip(lower=0).rolling(p).mean()
#         - df["Close"].diff().clip(upper=0).abs().rolling(p).mean()
#     ),

#     # ===== 10. FRACTAL ROUGHNESS PROXY =====
#     "roughness": lambda df, p: (
#         df["Close"].diff().abs().rolling(p).sum() /
#         (abs(df["Close"] - df["Close"].shift(p)) + 1e-12)
#     ),
# }


# def build_signals(data):

#     out = {}

#     for t, df in data.items():

#         df = compute_features(df)

#         sigs = {}

#         # ===== LOCKED SIGNALS (NO PERIOD LOOP) =====
#         for name, fn in LOCKED_SIGNALS.items():
#             try:
#                 sigs[name] = fn(df)
#             except:
#                 continue

#         # ===== NEW SIGNALS (SEARCH OVER PERIODS) =====
#         for name, fn in NEW_SIGNAL_LIBRARY.items():
#             for p in PERIODS:
#                 try:
#                     sigs[f"{name}_{p}"] = fn(df, p)
#                 except:
#                     continue

#         out[t] = pd.concat(sigs, axis=1)

#     return out

In [ ]:
def stack_signals(sig_dict):
    panels = []

    for t, df in sig_dict.items():
        df = df.copy()
        df["ticker"] = t
        df = df.set_index("ticker", append=True)
        panels.append(df)

    panel = pd.concat(panels).sort_index()
    panel.index.names = ["open_time","ticker"]

    return panel

In [ ]:
def pivot_signal(sig):
    return sig.reset_index().pivot(index="open_time", columns="ticker", values="score")

def normalize_signal(signal_df, method="rank"):
    """
    Normalize cross-section per timestamp.

    method:
        - "rank": percentile rank in [-0.5, 0.5]
        - "zscore": cross-sectional z-score
    """

    if method == "rank":
        ranked = signal_df.rank(axis=1, pct=True)
        return ranked - 0.5

    elif method == "zscore":
        return signal_df.sub(signal_df.mean(axis=1), axis=0).div(
            signal_df.std(axis=1) + 1e-12, axis=0
        )

    else:
        raise ValueError(f"Unknown normalization method: {method}")

def construct_portfolio(signal, top_k=3):
    w = pd.DataFrame(0.0, index=signal.index, columns=signal.columns)

    for t in signal.index:
        row = signal.loc[t].dropna()
        if len(row) < 2:
            continue

        idx = row.nlargest(top_k).index.union(row.nsmallest(top_k).index)
        row = row.loc[idx]

        weights = row / (np.abs(row).sum() + 1e-12)
        w.loc[t, weights.index] = weights

    return w.fillna(0.0)


def backtest(ret, w):
    strat = (w.shift(1) * ret).sum(axis=1)
    eq = (1 + strat).cumprod()
    sh = strat.mean() / (strat.std() + 1e-12) * np.sqrt(252)
    return strat, eq, float(sh)

In [ ]:
def construct_portfolio_topk(signal, top_k):

    w = pd.DataFrame(0.0, index=signal.index, columns=signal.columns)

    for t in signal.index:
        row = signal.loc[t].dropna()
        if len(row) < 2:
            continue

        idx = row.nlargest(top_k).index.union(row.nsmallest(top_k).index)
        row = row.loc[idx]

        weights = row / (np.abs(row).sum() + 1e-12)
        w.loc[t, weights.index] = weights

    return w.fillna(0.0)

In [ ]:
def eval_single(panel, ret, idx, name, top_k):

    s = panel[name]

    sig_df = s.rename("score").reset_index()
    sig_mat = pivot_signal(sig_df).loc[idx]

    # normalize cross-section
    sig_mat = normalize_signal(sig_mat, NORMALIZATION_METHOD)

    # ✅ top_k ONLY HERE
    w = construct_portfolio_topk(sig_mat, top_k)

    _, _, sh = backtest(ret.loc[idx], w)

    return sh


def evaluate_signals(panel, ret, train_idx):

    results = []

    # 🔥 extract base signal names
    base_signals = set()

    for c in panel.columns:
        parts = c.rsplit("_", 1)
        if parts[-1].isdigit():
            base_signals.add(parts[0])
        else:
            base_signals.add(c)  # static signal

    for base in tqdm(base_signals, desc="Signal Eval"):

        best = None

        for p in PERIODS:

            col = f"{base}_{p}"
            if col not in panel.columns:
                continue

            for k in TOP_K_LIST:

                sh = eval_single(panel, ret, train_idx, col, k)

                if best is None or abs(sh) > abs(best["sh"]):
                    best = {
                        "signal": base,
                        "period": p,
                        "top_k": k,
                        "sh": sh
                    }

        if best is not None:
            results.append(best)

    return pd.DataFrame(results).sort_values("sh", key=np.abs, ascending=False)

In [ ]:
def signal_to_returns(signal_series, ret_df, idx, top_k):

    sig_df = signal_series.rename("score").reset_index()
    sig_mat = pivot_signal(sig_df).loc[idx]

    sig_mat = normalize_signal(sig_mat, NORMALIZATION_METHOD)

    # ✅ APPLY top_k HERE (signal-level)
    w = construct_portfolio_topk(sig_mat, top_k)

    strat, _, _ = backtest(ret_df.loc[idx], w)

    return strat

In [ ]:
def build_signal_returns(pool, meta, ret_df, idx):

    returns = {}

    for name in pool:

        strat = signal_to_returns(
            pool[name],
            ret_df,
            idx,
            meta[name]
        )

        returns[name] = strat

    return pd.DataFrame(returns)

In [ ]:
def filter_signals(df, threshold=0.8):

    df = df.copy()
    df["abs_sh"] = df["sh"].abs()

    df = df[df["abs_sh"] >= threshold]

    # TRUE transformation
    df["flip"] = df["sh"].apply(lambda x: -1 if x < 0 else 1)

    return df


def build_signal_pool(panel, selected):

    pool = {}
    meta = {}

    for _, r in selected.iterrows():

        base   = r["signal"]
        period = int(r["period"])
        top_k  = int(r["top_k"])
        flip   = int(r["flip"])

        col = f"{base}_{period}"

        s = panel[col].copy()

        # 🔥 TRUE SIGN FLIP
        if flip == -1:
            s = -s

        pool[base] = s
        meta[base] = top_k

    return pool, meta

In [ ]:
def build_combined(pool, chosen, weights):

    combined = None

    for i, name in enumerate(chosen):
        s = pool[name] * weights[i]
        combined = s if combined is None else combined.add(s, fill_value=0)

    return combined


def fitness(chrom, signal_ret_df, idx):

    chosen, weights = chrom

    sub = signal_ret_df[chosen].loc[idx]

    # weighted combination of signals
    combined = (sub * weights).sum(axis=1)

    sharpe = combined.mean() / (combined.std()+1e-12) * np.sqrt(252)

    return sharpe


def run_ga(signal_ret_df, train_idx, val_idx, generations=30, pop_size=50):

    keys = list(signal_ret_df.columns)

    def rand():
        k = random.randint(2, min(6, len(keys)))
        chosen = random.sample(keys, k)

        w = np.random.rand(k)
        w /= w.sum()

        return (chosen, w)

    def compute_sharpe(series):
        return series.mean() / (series.std() + 1e-12) * np.sqrt(252)

    def fitness_full(chrom):
        chosen, weights = chrom

        train_sub = signal_ret_df[chosen].loc[train_idx]
        val_sub   = signal_ret_df[chosen].loc[val_idx]

        train_comb = (train_sub * weights).sum(axis=1)
        val_comb   = (val_sub * weights).sum(axis=1)

        sh_train = compute_sharpe(train_comb)
        sh_val   = compute_sharpe(val_comb)

        score = 0.7 * sh_train + 0.3 * sh_val

        return score, sh_train, sh_val

    def fmt_weights(w):
        return [round(float(x), 3) for x in w]

    population = [rand() for _ in range(pop_size)]

    # ✅ TRUE GLOBAL BEST TRACKING
    best = None
    best_score_global = -np.inf

    pbar = tqdm(range(generations), desc="GA Optimization")

    for g in pbar:

        scores = []
        metrics = []

        for c in population:
            score, sh_train, sh_val = fitness_full(c)
            scores.append(score)
            metrics.append((score, sh_train, sh_val))

        # ===== BEST IN CURRENT GENERATION =====
        i = int(np.argmax(scores))
        gen_best_score, gen_best_train, gen_best_val = metrics[i]
        gen_best_chosen, gen_best_weights = population[i]

        # ===== GLOBAL BEST UPDATE (FIXED) =====
        if gen_best_score > best_score_global:

            best_score_global = gen_best_score
            best = (
                gen_best_chosen,
                gen_best_weights,
                gen_best_score,
                gen_best_train,
                gen_best_val
            )

            print(
                f"GEN {g:02d} | "
                f"SCORE: {gen_best_score:.4f} | "
                f"TRAIN: {gen_best_train:.4f} | "
                f"VAL: {gen_best_val:.4f} | "
                f"N: {len(gen_best_chosen)} | "
                f"SIGNALS: {gen_best_chosen} | "
                f"W: {fmt_weights(gen_best_weights)}"
            )

        # ===== correct tqdm tracking =====
        pbar.set_postfix({
            "best_score": f"{best_score_global:.3f}",
            "train": f"{best[3]:.3f}",
            "val": f"{best[4]:.3f}"
        })

        # ===== selection =====
        probs = np.maximum(scores, 0)

        if probs.sum() == 0:
            probs = np.ones(len(scores)) / len(scores)
        else:
            probs = probs / probs.sum()

        new_pop = []

        for _ in range(pop_size):

            p1, p2 = random.choices(population, weights=probs, k=2)

            new_keys = list(set(p1[0]) | set(p2[0]))

            if len(new_keys) < 2:
                new_keys = random.sample(keys, 2)

            w = np.random.rand(len(new_keys))
            w /= w.sum()

            new_pop.append((new_keys, w))

        population = new_pop

    return best

In [ ]:
# =========================
# LOAD + ALIGN
# =========================
raw = align_data(load_full_data(tickers, "1d"))
ret = compute_returns(raw)

signals = build_signals(raw)
panel = stack_signals(signals)

# =========================
# ALIGNMENT
# =========================
valid = ret.index.intersection(panel.index.get_level_values(0))

panel = panel.loc[panel.index.get_level_values(0).isin(valid)]
ret = ret.loc[valid]

# =========================
# SPLIT
# =========================
n = len(ret)

train_end = int(n * 0.7)
val_end   = int(n * 0.85)

train = ret.index[:train_end]
val   = ret.index[train_end:val_end]
test  = ret.index[val_end:]

trainval = ret.index[:val_end]
all  = ret.index

print(f"""
DATA SPLIT:
Train:    {len(train)}
Val:      {len(val)}
Test:     {len(test)}
TrainVal: {len(trainval)}
All:      {len(all)}
""")

# =========================
# SIGNAL EVALUATION (TRAIN ONLY)
# =========================
perf = evaluate_signals(panel, ret, train)
display(perf)

# =========================
# FILTER SIGNALS
# =========================
selected = filter_signals(perf, 0.8)
display(selected)

# =========================
# BUILD SIGNAL POOL
# =========================
pool, meta = build_signal_pool(panel, selected)

# =========================
# 🔥 BUILD SIGNAL RETURNS
# =========================

# train+val → for GA optimization
signal_ret_trainval = build_signal_returns(pool, meta, ret, trainval)

# full → for final evaluation
signal_ret_all = build_signal_returns(pool, meta, ret, all)

# =========================
# GA OPTIMIZATION
# =========================
best = run_ga(signal_ret_trainval, train, val)

print("\nBEST CONFIG:")
print(best)


DATA SPLIT:
Train:    1235
Val:      265
Test:     265
TrainVal: 1500
All:      1765



Signal Eval:   0%|          | 0/33 [00:00<?, ?it/s]

,signal,period,top_k,sh
21,momentum,7,3,1.792268
23,rel_momentum,3,5,1.608712
26,range_reversion,21,5,1.534286
17,range_momentum,7,3,1.510754
30,volume_thrust,60,1,1.388704
4,range_vol_regime,21,2,1.288070
24,roughness,91,5,-1.285662
31,skewness,273,4,-1.254448
9,shock_recovery,91,5,-1.229917
18,close_pos_change,91,5,1.171397


,signal,period,top_k,sh,abs_sh,flip
21,momentum,7,3,1.792268,1.792268,1
23,rel_momentum,3,5,1.608712,1.608712,1
26,range_reversion,21,5,1.534286,1.534286,1
17,range_momentum,7,3,1.510754,1.510754,1
30,volume_thrust,60,1,1.388704,1.388704,1
4,range_vol_regime,21,2,1.288070,1.288070,1
24,roughness,91,5,-1.285662,1.285662,-1
31,skewness,273,4,-1.254448,1.254448,-1
9,shock_recovery,91,5,-1.229917,1.229917,-1
18,close_pos_change,91,5,1.171397,1.171397,1


GA Optimization:   0%|          | 0/30 [00:00<?, ?it/s]

GEN 00 | SCORE: 2.4170 | TRAIN: 2.6480 | VAL: 1.8779 | N: 6 | SIGNALS: ['roughness', 'momentum', 'vol_up_down_asym', 'vol_cluster', 'close_pos_change', 'skewness'] | W: [0.231, 0.205, 0.095, 0.049, 0.302, 0.118]
GEN 01 | SCORE: 2.5359 | TRAIN: 3.0598 | VAL: 1.3133 | N: 10 | SIGNALS: ['vol_cluster', 'shock_recovery', 'rel_momentum', 'wick_ratio', 'break_intensity', 'entropy', 'kurtosis', 'volume_thrust', 'momentum', 'vol_up_down_asym'] | W: [0.113, 0.061, 0.08, 0.115, 0.141, 0.155, 0.106, 0.017, 0.171, 0.04]
GEN 02 | SCORE: 2.7115 | TRAIN: 3.0581 | VAL: 1.9026 | N: 16 | SIGNALS: ['trend_stability', 'vol_cluster', 'wick_ratio', 'vol_regime', 'range_momentum', 'close_pos_change', 'kurtosis', 'momentum', 'vol_up_down_asym', 'shock_recovery', 'roughness', 'range_reversion', 'break_intensity', 'entropy', 'volume_thrust', 'skewness'] | W: [0.004, 0.105, 0.095, 0.063, 0.1, 0.036, 0.116, 0.011, 0.021, 0.044, 0.017, 0.063, 0.071, 0.071, 0.077, 0.106]
GEN 03 | SCORE: 2.8739 | TRAIN: 3.1322 | VAL:

In [ ]:
# =========================
# FINAL EVALUATION (ALL SPLITS)
# =========================

def eval_final(signal_ret_df, idx, best):
    chosen, weights = best[0], best[1]

    sub = signal_ret_df[chosen].loc[idx]

    combined = (sub * weights).sum(axis=1)

    sharpe = combined.mean() / (combined.std() + 1e-12) * np.sqrt(252)

    eq = (1 + combined).cumprod()

    return combined, eq, sharpe


# ===== compute ALL splits =====
train_net, train_eq, train_sh       = eval_final(signal_ret_all, train, best)
val_net, val_eq, val_sh             = eval_final(signal_ret_all, val, best)
test_net, test_eq, test_sh          = eval_final(signal_ret_all, test, best)
trainval_net, trainval_eq, trainval_sh = eval_final(signal_ret_all, trainval, best)
all_net, all_eq, all_sh             = eval_final(signal_ret_all, all, best)

# =========================
# PRINT SUMMARY
# =========================
print("\n===== SHARPE SUMMARY =====")
print(f"Train:      {train_sh:.4f}")
print(f"Validation: {val_sh:.4f}")
print(f"Test:       {test_sh:.4f}")
print(f"Train+Val:  {trainval_sh:.4f}")
print(f"Full:       {all_sh:.4f}")


===== SHARPE SUMMARY =====
Train:      3.1322
Validation: 2.2714
Test:       0.9085
Train+Val:  2.9654
Full:       2.7270


# PLOTS

In [ ]:
import plotly.graph_objects as go
import plotly.express as px

# =========================
# EQUITY CURVE (STITCHED)
# =========================

val_eq_adj = val_eq * train_eq.iloc[-1]
test_eq_adj = test_eq * val_eq_adj.iloc[-1]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=train_eq.index, y=train_eq,
    name="Train",
    line=dict(color='blue')
))

fig.add_trace(go.Scatter(
    x=val_eq_adj.index, y=val_eq_adj,
    name="Validation",
    line=dict(color='orange')
))

fig.add_trace(go.Scatter(
    x=test_eq_adj.index, y=test_eq_adj,
    name="Test",
    line=dict(color='green')
))

# Add vertical lines to separate the periods
fig.update_layout(
    shapes=[
        # Train-Validation boundary
        dict(
            type="line",
            x0=train_eq.index[-1],
            y0=0,
            x1=train_eq.index[-1],
            y1=1,
            yref='paper',
            line=dict(
                color="red",
                width=2,
                dash="dash"
            )
        ),
        # Validation-Test boundary
        dict(
            type="line",
            x0=val_eq_adj.index[-1],
            y0=0,
            x1=val_eq_adj.index[-1],
            y1=1,
            yref='paper',
            line=dict(
                color="red",
                width=2,
                dash="dash"
            )
        )
    ],
    annotations=[
        dict(
            x=train_eq.index[len(train_eq.index)//2],
            y=1.05,
            xref="x",
            yref="paper",
            text="Train",
            showarrow=False,
            font=dict(color="blue")
        ),
        dict(
            x=val_eq_adj.index[len(val_eq_adj.index)//2],
            y=1.05,
            xref="x",
            yref="paper",
            text="Validation",
            showarrow=False,
            font=dict(color="orange")
        ),
        dict(
            x=test_eq_adj.index[len(test_eq_adj.index)//2],
            y=1.05,
            xref="x",
            yref="paper",
            text="Test",
            showarrow=False,
            font=dict(color="green")
        )
    ],
    title="Equity Curve (Train → Val → Test)",
    xaxis_title="Time",
    yaxis_title="Equity",
    template="plotly_dark",
    height=600
)

fig.show()

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=all_eq.index,
    y=all_eq.values,
    name="Full Equity"
))

fig.update_layout(
    title="Full Sample Equity Curve",
    template="plotly_dark",
    height=500
)

fig.show()

In [ ]:
all_returns = pd.concat([val_net, test_net])

fig = px.histogram(
    all_returns,
    nbins=50,
    title="Return Distribution (OOS)",
    template="plotly_dark"
)

fig.show()

In [ ]:
def drawdown(equity):
    peak = equity.cummax()
    return equity / peak - 1

# Calculate drawdown for the entire equity curve
all_dd = drawdown(all_eq)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=all_dd.index,
    y=all_dd,
    name="Full Sample Drawdown"
))

fig.update_layout(
    shapes=[
        # Train-Validation boundary
        dict(
            type="line",
            x0=train_eq.index[-1],
            y0=all_dd.min(),
            x1=train_eq.index[-1],
            y1=all_dd.max(),
            line=dict(
                color="red",
                width=2,
                dash="dash"
            )
        ),
        # Validation-Test boundary
        dict(
            type="line",
            x0=val_eq_adj.index[-1],
            y0=all_dd.min(),
            x1=val_eq_adj.index[-1],
            y1=all_dd.max(),
            line=dict(
                color="red",
                width=2,
                dash="dash"
            )
        )
    ],
    annotations=[
        dict(
            x=train_eq.index[len(train_eq.index)//2],
            y=all_dd.max(),
            xref="x",
            yref="y",
            text="Train",
            showarrow=False,
            font=dict(color="blue")
        ),
        dict(
            x=val_eq_adj.index[len(val_eq_adj.index)//2],
            y=all_dd.max(),
            xref="x",
            yref="y",
            text="Validation",
            showarrow=False,
            font=dict(color="orange")
        ),
        dict(
            x=test_eq_adj.index[len(test_eq_adj.index)//2],
            y=all_dd.max(),
            xref="x",
            yref="y",
            text="Test",
            showarrow=False,
            font=dict(color="green")
        )
    ],
    title="Drawdown Curve (Full Sample)",
    template="plotly_dark",
    height=500
)

fig.show()

In [ ]:
all_eq = all_eq.sort_index()

def drawdown(equity):
    peak = equity.cummax()
    return equity / peak - 1

all_dd = drawdown(all_eq)

dd_train = all_dd.loc[train_eq.index]
dd_val = all_dd.loc[val_eq.index]
dd_test = all_dd.loc[test_eq.index]
dd_trainval = all_dd.loc[trainval_eq.index]

def dd_stats(dd):
    return {
        "max_dd": dd.min(),
        "final_dd": dd.iloc[-1],
        "mean_dd": dd.mean()
    }

results = {
    "train": dd_stats(dd_train),
    "val": dd_stats(dd_val),
    "test": dd_stats(dd_test),
    "trainval": dd_stats(dd_trainval),
    "full": dd_stats(all_dd)
}

for k, v in results.items():
    print(f"\n{k.upper()}")
    print(f"  Max DD   : {v['max_dd']:.4f}")
    print(f"  Final DD : {v['final_dd']:.4f}")
    print(f"  Mean DD  : {v['mean_dd']:.4f}")


TRAIN
  Max DD   : -0.0451
  Final DD : -0.0305
  Mean DD  : -0.0080

VAL
  Max DD   : -0.0451
  Final DD : -0.0174
  Mean DD  : -0.0183

TEST
  Max DD   : -0.0583
  Final DD : -0.0404
  Mean DD  : -0.0240

TRAINVAL
  Max DD   : -0.0451
  Final DD : -0.0174
  Mean DD  : -0.0098

FULL
  Max DD   : -0.0583
  Final DD : -0.0404
  Mean DD  : -0.0119


In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=all_dd.index,
    y=all_dd,
    name="Continuous Drawdown",
    line=dict(width=2)
))

# split markers (visual only)
fig.add_vline(x=train_eq.index[-1], line_dash="dash", line_color="red")
fig.add_vline(x=val_eq.index[-1], line_dash="dash", line_color="red")

fig.update_layout(
    title="Continuous Drawdown Curve (No Reset Across Splits)",
    template="plotly_dark",
    height=600
)

fig.show()

In [ ]:
fig = go.Figure(data=[
    go.Bar(name="Train", x=["Sharpe"], y=[train_sh]),
    go.Bar(name="Validation", x=["Sharpe"], y=[val_sh]),
    go.Bar(name="Test", x=["Sharpe"], y=[test_sh]),
    go.Bar(name="Train+Val", x=["Sharpe"], y=[trainval_sh]),
    go.Bar(name="Full Sample", x=["Sharpe"], y=[all_sh])
])

fig.update_layout(
    title="Sharpe Comparison",
    template="plotly_dark",
    barmode="group"
)

fig.show()